# BAN 8101 Predictive Analytics
## Week 9: Deep Neural Networks for Autonomous Driving

Dr.Junyao Yang

### Big Question
**What changes when a neural network becomes deeper, and how do we train it without simply overfitting the training data?**


# 1. From Week 8 to Week 9

Week 8 introduced the basic training cycle:

$$
\text{Input}\rightarrow\text{Forward Pass}\rightarrow\text{Prediction}\rightarrow\text{Loss}\rightarrow\text{Backpropagation}\rightarrow\text{Weight Update}
$$

Week 9 keeps that same cycle but increases network depth and training realism.

### Shallow example

$$
5\rightarrow16\rightarrow3
$$

### Deeper example

$$
5\rightarrow64\rightarrow32\rightarrow16\rightarrow3
$$

The deeper network has greater **capacity**, but greater capacity also creates more opportunity to overfit.

# 2. Model Capacity

Model capacity describes how flexible a model is.

More layers and neurons usually mean more trainable parameters. That can help the network learn complicated patterns, but it does not guarantee better performance on unseen data.

$$
\text{Too Little Capacity}\rightarrow\text{Underfitting}
$$

$$
\text{Too Much Capacity}\rightarrow\text{Overfitting Risk}
$$

The correct question is not **Which network is biggest?**

It is **Which network generalizes best?**

# 3. Create a Driving-State Dataset

In [1]:
# ---------------------------------------------------------------
# WHY THIS DATASET IS SYNTHETIC
#
# Week 9 is still focused on DNN concepts rather than image processing.
# Each row represents one driving moment with structured vehicle/road
# variables. This lets us isolate the effect of network depth,
# mini-batches, learning rate, and dropout.
# ---------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = 6000

# Structured driving variables. These are synthetic but resemble
# information an autonomous-driving system could receive.
road_curvature = rng.uniform(-1.0, 1.0, n)
lateral_offset = np.clip(rng.normal(0, 0.35, n), -1.2, 1.2)
heading_error = np.clip(rng.normal(0, 6.0, n), -20, 20)
speed = rng.uniform(5, 25, n)
obstacle_distance = rng.uniform(5, 100, n)

# Create a nonlinear steering score.
# The interaction terms give the neural networks meaningful
# nonlinear structure to learn.
steering_score = (
    0.95 * road_curvature
    - 0.55 * lateral_offset
    - 0.025 * heading_error
    + 0.35 * (road_curvature ** 3)
    + 0.015 * speed * road_curvature
    - 0.08 * lateral_offset * road_curvature
    + rng.normal(0, 0.14, n)
)

LEFT_THRESHOLD = -0.12
RIGHT_THRESHOLD = 0.12

steering_class = np.where(
    steering_score < LEFT_THRESHOLD,
    'Left',
    np.where(steering_score > RIGHT_THRESHOLD, 'Right', 'Straight')
)

driving = pd.DataFrame({
    'road_curvature': road_curvature,
    'lateral_offset': lateral_offset,
    'heading_error': heading_error,
    'speed': speed,
    'obstacle_distance': obstacle_distance,
    'steering_class': steering_class
})

driving.head()

# RESULT INTERPRETATION:
# driving.head() should show one row per driving moment.
# The values are artificial, so treat them as teaching data rather than
# measurements from a real vehicle.


,road_curvature,lateral_offset,heading_error,speed,obstacle_distance,steering_class
0,0.547912,-0.362948,0.959589,11.464480,99.962125,Right
1,-0.122243,0.168922,5.357384,16.976954,43.968306,Left
2,0.717196,-0.212635,3.634865,9.151898,54.630415,Right
3,0.394736,-0.577173,-1.211692,5.851390,66.859244,Right
4,-0.811645,-0.100615,6.194357,18.849093,92.453030,Left


### How to Read This Output

Each row represents one driving state. The predictors describe the current vehicle/road condition, and `steering_class` is the outcome to predict.

Because these data are synthetic, focus on the **modeling workflow**, not on whether the numerical values perfectly represent a physical vehicle.

In [2]:
# Check class balance before modeling.
display(pd.DataFrame({
    'Count': driving['steering_class'].value_counts(),
    'Proportion': driving['steering_class'].value_counts(normalize=True).round(3)
}))

,Count,Proportion
steering_class,,
Left,2804,0.467
Right,2638,0.440
Straight,558,0.093


## What to Notice

The data contain nonlinear patterns such as:

$$
speed\times road\ curvature
$$

and:

$$
lateral\ offset\times road\ curvature
$$

A deeper neural network can potentially learn these interactions without us manually specifying every interaction feature.

# 4. Prepare Training, Validation, and Test Data

In [3]:
# ---------------------------------------------------------------
# THREE-WAY DATA SPLIT
#
# Training   -> updates the neural-network parameters
# Validation -> compares architectures and tuning choices
# Test       -> final evaluation after development is complete
#
# This separation helps prevent overly optimistic conclusions.
# ---------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

feature_names = [
    'road_curvature',
    'lateral_offset',
    'heading_error',
    'speed',
    'obstacle_distance'
]

X = driving[feature_names].to_numpy(dtype=np.float32)
y_text = driving['steering_class'].to_numpy()

class_to_id = {'Left': 0, 'Straight': 1, 'Right': 2}
y = np.array([class_to_id[label] for label in y_text], dtype=np.int64)

# Final test set stays untouched during model development.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Validation data are used to compare architectures and detect overfitting.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, random_state=RANDOM_STATE, stratify=y_dev
)

print('Training:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

# RESULT INTERPRETATION:
# The printed shapes show how many observations are assigned to each set.
# Training should be the largest group.


Training: (3840, 5)
Validation: (960, 5)
Test: (1200, 5)


In [4]:
# ---------------------------------------------------------------
# WHY WE STANDARDIZE
#
# Variables such as speed and obstacle distance use different numerical
# scales. Standardization puts them on more comparable scales, which
# usually helps gradient-based optimization.
# ---------------------------------------------------------------

# Standardize using TRAINING data only to avoid leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print('Training means after scaling:')
print(np.round(X_train_scaled.mean(axis=0), 3))

# RESULT INTERPRETATION:
# The training means should be very close to 0 after standardization.


Training means after scaling:
[-0. -0. -0.  0. -0.]


# 5. Mini-Batch Training

In Week 8, we could train using the entire training set at once.

Deep neural networks are commonly trained using **mini-batches**.

Instead of one update from the full dataset, the network may update after every 64 observations.

The number of observations used for one update is the **batch size**.

In [5]:
# ---------------------------------------------------------------
# MINI-BATCH TRAINING
#
# DataLoader breaks the training data into smaller groups.
# With BATCH_SIZE = 64, each optimizer update uses 64 observations.
# ---------------------------------------------------------------

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# TensorDataset pairs each input row with its class label.
train_dataset = TensorDataset(X_train_t, y_train_t)

# DataLoader creates mini-batches and shuffles their order each epoch.
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print('Training observations:', len(train_dataset))
print('Mini-batches per epoch:', len(train_loader))

# RESULT INTERPRETATION:
# 'Mini-batches per epoch' tells you roughly how many parameter updates
# occur during one complete pass through the training data.


ModuleNotFoundError: No module named 'torch'

### How to Interpret Mini-Batches

If the notebook reports about 60 mini-batches per epoch, the optimizer performs about 60 weight updates during each epoch.

That is one major difference from full-batch training: learning happens through many smaller updates.

## Check Your Understanding

If there are 3,840 training observations and the batch size is 64:

$$
3840/64=60
$$

The optimizer updates the network about 60 times during one epoch.

# 6. Compare Three Network Architectures

### Small

$$
5\rightarrow8\rightarrow3
$$

### Medium

$$
5\rightarrow32\rightarrow16\rightarrow3
$$

### Deep

$$
5\rightarrow64\rightarrow32\rightarrow16\rightarrow3
$$

In [ ]:
# ---------------------------------------------------------------
# SAME PREDICTION TASK, DIFFERENT MODEL CAPACITY
#
# Input dimension = 5 for all models.
# Output dimension = 3 steering classes for all models.
# Only the hidden structure changes.
# ---------------------------------------------------------------

class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 8),
            nn.ReLU(),
            nn.Linear(8, 3)
        )
    def forward(self, x):
        return self.network(x)


class MediumNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.network(x)


class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.network(x)

# HOW TO READ nn.Linear(a, b):
# a = number of values entering the layer
# b = number of values leaving the layer


# 7. Count Trainable Parameters

In [ ]:
# ---------------------------------------------------------------
# TRAINABLE PARAMETER COUNT
#
# Every learned weight and bias is a parameter.
# More parameters generally mean more capacity and more computation.
# ---------------------------------------------------------------

def count_parameters(model):
    # Count trainable weights and biases.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

parameter_table = pd.DataFrame({
    'Model': ['Small', 'Medium', 'Deep'],
    'Trainable Parameters': [
        count_parameters(SmallNet()),
        count_parameters(MediumNet()),
        count_parameters(DeepNet())
    ]
})

parameter_table

# RESULT INTERPRETATION:
# Use parameter count as a complexity measure, not as a quality score.


## Pause and Think

More parameters mean more flexibility, but not automatically better generalization.

The larger model must earn its complexity through validation performance or another meaningful advantage.

# 8. Learning Rate

The learning rate controls the size of the parameter updates:

$$
w_{new}=w_{old}-\eta\frac{\partial L}{\partial w}
$$

- Too large: training may become unstable.
- Too small: training may improve very slowly.

# 9. Reusable Mini-Batch Training Function

In [ ]:
# ---------------------------------------------------------------
# REUSABLE TRAINING FUNCTION
#
# We use one function for all networks so the comparison is fair.
# Each architecture follows the same training process.
# ---------------------------------------------------------------

def train_model(model, train_loader, X_val_t, y_val_t, epochs=80, learning_rate=0.01):
    # Multiclass loss.
    criterion = nn.CrossEntropyLoss()

    # Adam updates weights using gradients from backpropagation.
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_history = []
    val_history = []

    for epoch in range(epochs):
        # ---------------- TRAINING ----------------
        model.train()
        total_loss = 0.0
        total_n = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()       # clear previous gradients
            logits = model(X_batch)     # forward pass
            loss = criterion(logits, y_batch)
            loss.backward()             # backpropagation
            optimizer.step()            # update parameters

            batch_n = X_batch.shape[0]
            total_loss += loss.item() * batch_n
            total_n += batch_n

        train_history.append(total_loss / total_n)

        # ---------------- VALIDATION ----------------
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t)

        val_history.append(val_loss.item())

    return train_history, val_history

# IMPORTANT:
# We return both training and validation losses because a low training
# loss alone does not prove good generalization.


# 10. Train All Three Architectures

In [ ]:
# ---------------------------------------------------------------
# FAIR ARCHITECTURE COMPARISON
#
# Small, Medium, and Deep use the same number of epochs and the same
# learning rate. This isolates the effect of architecture size.
# ---------------------------------------------------------------

EPOCHS = 80
LEARNING_RATE = 0.01

torch.manual_seed(RANDOM_STATE)
small_model = SmallNet()

torch.manual_seed(RANDOM_STATE)
medium_model = MediumNet()

torch.manual_seed(RANDOM_STATE)
deep_model = DeepNet()

small_train_loss, small_val_loss = train_model(
    small_model, train_loader, X_val_t, y_val_t, EPOCHS, LEARNING_RATE
)

medium_train_loss, medium_val_loss = train_model(
    medium_model, train_loader, X_val_t, y_val_t, EPOCHS, LEARNING_RATE
)

deep_train_loss, deep_val_loss = train_model(
    deep_model, train_loader, X_val_t, y_val_t, EPOCHS, LEARNING_RATE
)

print('Training complete.')

# EXPECTATION:
# The deepest network is not guaranteed to have the best validation loss.


# 11. Compare Validation Loss

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(small_val_loss, label='Small')
plt.plot(medium_val_loss, label='Medium')
plt.plot(deep_val_loss, label='Deep')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss by Architecture')
plt.legend()
plt.show()

# HOW TO READ THIS PLOT:
# - Lower validation loss is generally better.
# - A curve that falls and then rises may indicate overfitting.
# - A consistently high curve may indicate underfitting.


### How to Read the Validation-Loss Curves

Look for:

- which model reaches the lowest validation loss,
- whether any curve begins increasing after initially improving,
- whether the deeper network actually provides a meaningful advantage.

A model can fit the training data extremely well and still generalize poorly.

## What to Notice

The best training fit is not necessarily the best validation fit.

The central question is:

> **Which architecture generalizes best to unseen data?**

# 12. Compare Validation Accuracy and Macro F1

In [ ]:
# ---------------------------------------------------------------
# TURN MODEL OUTPUTS INTO FAMILIAR METRICS
#
# Accuracy = overall fraction classified correctly.
# Macro F1 = gives Left, Straight, and Right equal importance.
# ---------------------------------------------------------------

def evaluate_model(model, X_tensor, y_true):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        pred = torch.argmax(logits, dim=1).numpy()

    accuracy = accuracy_score(y_true, pred)
    macro_f1 = f1_score(y_true, pred, average='macro')
    return accuracy, macro_f1, pred

results = []
predictions = {}

for name, model in {
    'Small': small_model,
    'Medium': medium_model,
    'Deep': deep_model
}.items():
    acc, f1, pred = evaluate_model(model, X_val_t, y_val)
    predictions[name] = pred
    results.append({
        'Model': name,
        'Validation Accuracy': acc,
        'Validation Macro F1': f1,
        'Parameters': count_parameters(model)
    })

comparison = pd.DataFrame(results)
comparison.round(3)

# RESULT INTERPRETATION:
# Compare validation metrics together with parameter count.
# A tiny gain from a much larger network may not justify the complexity.


### Interpreting the Architecture Comparison

Use the validation metrics together with model size.

For example, if the Medium and Deep networks have nearly identical Macro F1 but the Medium model uses far fewer parameters, the Medium network may be the more defensible choice.

If the Deep network produces a substantial validation improvement, the additional complexity may be justified.

# 13. Underfitting vs. Overfitting

### Underfitting
Training and validation performance are both weak.

### Good Generalization
Training and validation performance are both strong.

### Overfitting
Training performance keeps improving while validation performance stops improving or becomes worse.

# 14. Dropout

Dropout is a regularization technique. During training, it randomly deactivates some hidden-unit outputs.

The goal is to reduce over-reliance on individual neurons.

Dropout is active during `model.train()` and disabled during `model.eval()`.

In [ ]:
# ---------------------------------------------------------------
# DROPOUT REGULARIZATION
#
# p=0.20 means roughly 20% of selected hidden activations are randomly
# set to zero during training. This discourages the model from depending
# too heavily on particular hidden units.
# ---------------------------------------------------------------

class DeepNetWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.network(x)


torch.manual_seed(RANDOM_STATE)
dropout_model = DeepNetWithDropout()

drop_train_loss, drop_val_loss = train_model(
    dropout_model, train_loader, X_val_t, y_val_t, EPOCHS, LEARNING_RATE
)

drop_acc, drop_f1, drop_pred = evaluate_model(dropout_model, X_val_t, y_val)

print('Dropout Accuracy:', round(drop_acc, 3))
print('Dropout Macro F1:', round(drop_f1, 3))

# RESULT INTERPRETATION:
# Dropout is helpful only if it improves validation behavior or reduces
# overfitting. It is not guaranteed to improve every dataset.


### Interpreting Dropout

Do not assume dropout should always raise accuracy.

Its purpose is to improve generalization by reducing over-reliance on particular hidden units. Compare validation metrics and loss curves before deciding whether dropout helped.

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(deep_val_loss, label='Deep')
plt.plot(drop_val_loss, label='Deep + Dropout')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Effect of Dropout')
plt.legend()
plt.show()

# HOW TO READ THIS PLOT:
# Compare validation loss with and without dropout.
# Lower or more stable validation loss can suggest better generalization.


### What to Notice

If dropout produces lower or more stable validation loss, it may be reducing overfitting.

If it performs worse, the original model may not need that amount of regularization, or the dropout rate may be too strong.

## Interpretation

Dropout is not guaranteed to improve every dataset.

The important question is whether regularization improves validation behavior enough to justify using it.

# 15. Learning-Rate Experiment

In [ ]:
# ---------------------------------------------------------------
# LEARNING-RATE EXPERIMENT
#
# The architecture stays fixed. Only the optimization step size changes.
# This helps us see how learning rate affects training behavior.
# ---------------------------------------------------------------

learning_rates = [0.1, 0.01, 0.001]
lr_histories = {}

for lr in learning_rates:
    torch.manual_seed(RANDOM_STATE)
    temp_model = MediumNet()
    _, temp_val_loss = train_model(
        temp_model,
        train_loader,
        X_val_t,
        y_val_t,
        epochs=40,
        learning_rate=lr
    )
    lr_histories[lr] = temp_val_loss

plt.figure(figsize=(9,5))
for lr, history in lr_histories.items():
    plt.plot(history, label=f'Learning Rate = {lr}')

plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Learning Rate Comparison')
plt.legend()
plt.show()

# RESULT INTERPRETATION:
# Large learning rates can be unstable.
# Very small learning rates may improve too slowly.


### How to Read the Learning-Rate Curves

A very large learning rate may cause unstable or oscillating loss. A very small learning rate may reduce loss slowly.

The goal is not simply the smallest learning rate; it is a value that learns efficiently and stably.

## Pause and Think

- Did the largest learning rate behave smoothly?
- Did the smallest learning rate improve slowly?
- Which value learned efficiently without becoming unstable?

# 16. Inspect a Confusion Matrix

In [ ]:
# ---------------------------------------------------------------
# CONFUSION MATRIX
#
# The diagonal shows correct predictions.
# Off-diagonal cells show specific steering mistakes.
# ---------------------------------------------------------------

medium_pred = predictions['Medium']

ConfusionMatrixDisplay.from_predictions(
    y_val,
    medium_pred,
    display_labels=['Left', 'Straight', 'Right'],
    cmap='Blues'
)

plt.title('Medium DNN - Validation Confusion Matrix')
plt.show()

# SELF-DRIVING INTERPRETATION:
# Opposite-direction mistakes may be more serious than a mild
# Left/Straight or Right/Straight confusion.


### How to Read the Confusion Matrix

The diagonal cells are correct predictions. Off-diagonal cells are mistakes.

For autonomous driving, the type of mistake matters. A model that predicts Right when the correct class is Left may be more concerning than a model that predicts Straight for a mild turn.

## Self-Driving Interpretation

Do not look only at overall accuracy.

Ask:
- Does the model miss turns?
- Does it introduce unnecessary turns?
- Does it ever predict the opposite direction?

As we move closer to the simulator project, the **type of error** becomes increasingly important.

# 17. Save a Trained PyTorch Model

In the final project, the simulator should load a trained model rather than retrain it every time it starts.

$$
\text{Train}\rightarrow\text{Save}\rightarrow\text{Load Later}\rightarrow\text{Predict}
$$

In [ ]:
# ---------------------------------------------------------------
# SAVE THE TRAINED PARAMETERS
#
# state_dict() stores learned weights and biases.
# This lets us reuse the trained network without retraining it.
# ---------------------------------------------------------------

from pathlib import Path

MODEL_PATH = Path('week9_medium_steering_model.pt')

# state_dict contains learned weights and biases.
torch.save(medium_model.state_dict(), MODEL_PATH)

print('Saved model:', MODEL_PATH)

# FINAL-PROJECT CONNECTION:
# The simulator should load a trained model and use it repeatedly for
# prediction rather than retraining at every time step.


### Why Saving the Model Matters

Training and deployment are different stages.

The network can be trained once, saved, and later loaded into a simulator-control program. This makes real-time prediction practical.

# 18. Reload the Saved Model

In [ ]:
# ---------------------------------------------------------------
# RELOAD THE SAVED NETWORK
#
# First recreate the same architecture, then load its saved parameters.
# ---------------------------------------------------------------

# Recreate the SAME architecture.
loaded_model = MediumNet()

# Load the saved parameters.
loaded_model.load_state_dict(
    torch.load(MODEL_PATH, weights_only=True)
)

loaded_model.eval()

loaded_acc, loaded_f1, loaded_pred = evaluate_model(
    loaded_model, X_val_t, y_val
)

print('Reloaded Accuracy:', round(loaded_acc, 3))
print('Reloaded Macro F1:', round(loaded_f1, 3))

# RESULT INTERPRETATION:
# Reloaded validation metrics should match the original model closely.


### Check Your Understanding

If the reloaded model reproduces the original model's validation performance, the learned parameters were successfully restored.

This is an important prerequisite for connecting a trained model to the simulator.

# 19. Why Saving Matters for the Simulator

Eventually, the workflow will be:

$$
\text{Train Model}
\rightarrow
\text{Save Model}
\rightarrow
\text{Launch Simulator}
\rightarrow
\text{Load Model}
\rightarrow
\text{Predict Repeatedly}
$$

The vehicle needs fast predictions, not repeated retraining.

# 20. From Static Prediction to Continuous Control

So far:

$$
\text{One Driving State}\rightarrow\text{One Steering Prediction}
$$

In the simulator:

$$
State_t\rightarrow Action_t\rightarrow State_{t+1}\rightarrow Action_{t+1}\rightarrow\cdots
$$

A prediction changes the vehicle's next state. This is a major difference between ordinary offline prediction and autonomous control.

# 21. Bridge to the Simulator Activity

The second Week 9 activity will introduce the simulator itself.

The full system will eventually look like:

$$
\text{Simulator Observation}
\rightarrow
\text{Model}
\rightarrow
\text{Steering / Throttle}
\rightarrow
\text{Simulator}
$$

Later weeks will add camera-image preprocessing, CNN steering prediction, simulator integration, and robustness testing.

# 22. Week 9 Takeaways

- Deeper networks have more model capacity.
- More capacity can improve learning but also increase overfitting risk.
- Mini-batches allow several weight updates per epoch.
- Batch size controls how many observations are used per update.
- Learning rate controls the size of parameter updates.
- Dropout can reduce over-reliance on individual hidden units.
- Validation evidence should guide architecture selection.
- A trained model can be saved and reloaded without retraining.
- Simulator control turns static prediction into a continuous decision process.

# Reflection Questions

1. What is the main difference between a shallow neural network and a deeper neural network?
2. Why can adding more layers or neurons improve training performance but hurt validation performance?
3. What does **batch size** control, and why do deep neural networks often use mini-batches?
4. What does the **learning rate** control? What can happen if it is too large or too small?
5. What does dropout attempt to accomplish during neural-network training?
6. Based on the architecture comparison, which network would you select and why? Use validation evidence.
7. Why is saving and reloading a trained model important for the final self-driving simulator project?
8. What new challenge appears when we move from independent predictions to continuously controlling a vehicle?